# Test Device Evolution với Multiple Thresholds

## Mục tiêu
- Load data từ `evo_attribute_details.csv`
- Group theo **AccountID**
- Tính similarity score giữa các fingerprints với weights: 63, 17, 12, 6, 2
- Test với thresholds: **70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90**
- Đánh giá performance với metrics: Precision, Recall, F1-Score, Accuracy

## Weights Configuration
- **A_Hardware**: 63 điểm (63%)
- **D_Browser**: 17 điểm (17%)
- **E_Preferences**: 12 điểm (12%)
- **C_Audio**: 6 điểm (6%)
- **B_Graphic**: 2 điểm (2%)
- **Tổng**: 100 điểm

## Test Thresholds
- 80 điểm (80%)
- 81 điểm (81%) ← Recommended
- 82 điểm (82%)


## Import Libraries và Configuration


In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration
GROUP_WEIGHTS = {
    'A_Hardware': 19.52,
    'D_Browser': 10.72,
    'E_Preferences': 22.92,
    'C_Audio': 18.23,
    'B_Graphic': 28.61
}
TOTAL_WEIGHT = sum(GROUP_WEIGHTS.values())

TEST_THRESHOLDS = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69 ,70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80]
DATA_FILE = '../data/evo_attribute_details_test_account.csv'

GROUP_COLUMNS = {
    'A_Hardware': 'HardwareCode',
    'B_Graphic': 'GraphicCode',
    'C_Audio': 'AudioCode',
    'D_Browser': 'BrowserCode',
    'E_Preferences': 'PreferencesCode'
}

print("="*80)
print("DEVICE EVOLUTION THRESHOLD TESTING")
print("="*80)
print(f"\n✓ Configuration:")
print(f"  - Group Weights: {GROUP_WEIGHTS}")
print(f"  - Total Weight: {TOTAL_WEIGHT}")
print(f"  - Test Thresholds: {TEST_THRESHOLDS}")


DEVICE EVOLUTION THRESHOLD TESTING

✓ Configuration:
  - Group Weights: {'A_Hardware': 19.52, 'D_Browser': 10.72, 'E_Preferences': 22.92, 'C_Audio': 18.23, 'B_Graphic': 28.61}
  - Total Weight: 100.0
  - Test Thresholds: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80]


## Load Data


In [2]:
df = pd.read_csv(DATA_FILE, encoding='latin1')

print(f"✓ Data loaded: {len(df):,} records")
print(f"✓ Unique AccountIDs: {df['AccountID'].nunique():,}")
print(f"✓ Unique DeviceCodeIDs: {df['DeviceCodeID'].nunique():,}")

# Tạo hash columns
for group_name, col_name in GROUP_COLUMNS.items():
    if col_name in df.columns:
        df[f'{group_name}'] = df[col_name].astype(str).fillna('')

print(f"\n✓ Hash columns created for {len(GROUP_COLUMNS)} groups")
print(f"\nSample data:")
df[['AccountID', 'DeviceCodeID'] + [f'{g}' for g in GROUP_COLUMNS.keys()]].head()


✓ Data loaded: 5,000 records
✓ Unique AccountIDs: 3,644
✓ Unique DeviceCodeIDs: 2,096

✓ Hash columns created for 5 groups

Sample data:


,AccountID,DeviceCodeID,A_Hardware,B_Graphic,C_Audio,D_Browser,E_Preferences
0,59142370,19868,49ba84d249ba84d249ba84d249ba84d2,750e5ec5750e5ec5750e5ec5750e5ec5,e46d7fbce46d7fbce46d7fbce46d7fbc,b3b91d14b3b91d14b3b91d14b3b91d14,8ac2cbd78ac2cbd78ac2cbd78ac2cbd7
1,36449486,19868,49ba84d249ba84d249ba84d249ba84d2,750e5ec5750e5ec5750e5ec5750e5ec5,e46d7fbce46d7fbce46d7fbce46d7fbc,b3b91d14b3b91d14b3b91d14b3b91d14,8ac2cbd78ac2cbd78ac2cbd78ac2cbd7
2,59142370,19868,49ba84d249ba84d249ba84d249ba84d2,92f1830d92f1830d92f1830d92f1830d,e46d7fbce46d7fbce46d7fbce46d7fbc,b3b91d14b3b91d14b3b91d14b3b91d14,8ac2cbd78ac2cbd78ac2cbd78ac2cbd7
3,93299854,2476633,824a6431824a6431824a6431824a6431,536f4566536f4566536f4566536f4566,f02c062df02c062df02c062df02c062d,b3b91d14b3b91d14b3b91d14b3b91d14,b34f7818b34f7818b34f7818b34f7818
4,32975063,3746888,6e9d807b6e9d807b6e9d807b6e9d807b,9a538e6d9a538e6d9a538e6d9a538e6d,85f8bd6385f8bd6385f8bd6385f8bd63,13a6e41c13a6e41c13a6e41c13a6e41c,3ac76f623ac76f623ac76f623ac76f62


## Định nghĩa hàm tính Similarity Score


In [3]:
def calculate_similarity_score(fp1, fp2, weights):
    """
    Tính similarity score giữa 2 fingerprints
    
    Args:
        fp1: Series/row của fingerprint 1
        fp2: Series/row của fingerprint 2
        weights: Dict weights của từng group
    
    Returns:
        dict: {
            'total_score': int,      # Tổng điểm (0-100)
            'match_rate': float,     # Tỷ lệ match (0-1)
            'matched_groups': list,  # Danh sách groups đã match
            'unmatched_groups': list # Danh sách groups không match
        }
    """
    total_score = 0
    matched_groups = []
    unmatched_groups = []
    
    for group_name, weight in weights.items():
        hash_col = f'{group_name}'
        hash1 = str(fp1.get(hash_col, ''))
        hash2 = str(fp2.get(hash_col, ''))
        
        # Clean up
        hash1 = '' if hash1 in ['nan', 'None', ''] else hash1
        hash2 = '' if hash2 in ['nan', 'None', ''] else hash2
        
        # Match nếu hash giống nhau và không rỗng
        if hash1 == hash2 and hash1 != '':
            total_score += weight
            matched_groups.append(group_name)
        else:
            unmatched_groups.append(group_name)
    
    match_rate = total_score / TOTAL_WEIGHT if TOTAL_WEIGHT > 0 else 0
    
    return {
        'total_score': total_score,
        'match_rate': match_rate,
        'matched_groups': matched_groups,
        'unmatched_groups': unmatched_groups
    }

print("✓ Similarity score function defined")


✓ Similarity score function defined


## Tạo Validation Pairs (so sánh tất cả fingerprints trong cùng AccountID)


In [4]:
# Group theo AccountID
accounts = df.groupby('AccountID')

validation_pairs = []

print(f"Processing {len(accounts)} accounts...")

for account_id, account_df in accounts:
    # Sort theo index (giả sử thứ tự trong file là thứ tự thời gian)
    account_df = account_df.reset_index(drop=True).sort_index()
    
    # Chỉ xử lý accounts có >= 2 fingerprints
    if len(account_df) < 2:
        continue
    
    # Tạo pairs: so sánh fingerprint i với fingerprint j (i < j)
    for i in range(len(account_df)):
        for j in range(i + 1, len(account_df)):
            fp1 = account_df.iloc[i]
            fp2 = account_df.iloc[j]
            
            # Ground truth: cùng device nếu cùng DeviceCodeID
            is_same_device = (fp1['DeviceCodeID'] == fp2['DeviceCodeID'])
            
            # Tính similarity
            similarity = calculate_similarity_score(fp1, fp2, GROUP_WEIGHTS)
            
            validation_pairs.append({
                'AccountID': account_id,
                'fp1_index': i,
                'fp2_index': j,
                'fp1_DeviceCodeID': fp1['DeviceCodeID'],
                'fp2_DeviceCodeID': fp2['DeviceCodeID'],
                'ground_truth_same_device': is_same_device,
                'similarity_score': similarity['total_score'],
                'match_rate': similarity['match_rate'],
                'matched_groups': ','.join(similarity['matched_groups']),
                'unmatched_groups': ','.join(similarity['unmatched_groups']),
                'num_matched_groups': len(similarity['matched_groups'])
            })

validation_df = pd.DataFrame(validation_pairs)

print(f"\n✓ Generated {len(validation_df):,} validation pairs")
print(f"✓ Same device pairs: {validation_df['ground_truth_same_device'].sum():,}")
print(f"✓ Different device pairs: {(~validation_df['ground_truth_same_device']).sum():,}")

# Sample
print(f"\nSample pairs:")
validation_df.head(10)


Processing 3644 accounts...

✓ Generated 1,999 validation pairs
✓ Same device pairs: 1,054
✓ Different device pairs: 945

Sample pairs:


,AccountID,fp1_index,fp2_index,fp1_DeviceCodeID,fp2_DeviceCodeID,ground_truth_same_device,similarity_score,match_rate,matched_groups,unmatched_groups,num_matched_groups
0,5154,0,1,1422388674,1422388674,True,71.39,0.7139,"A_Hardware,D_Browser,E_Preferences,C_Audio",B_Graphic,4
1,7880,0,1,730860912,730860912,True,89.28,0.8928,"A_Hardware,E_Preferences,C_Audio,B_Graphic",D_Browser,4
2,9887,0,1,1419247292,1419247292,True,71.39,0.7139,"A_Hardware,D_Browser,E_Preferences,C_Audio",B_Graphic,4
3,9887,0,2,1419247292,1419247292,True,77.08,0.7708,"A_Hardware,D_Browser,C_Audio,B_Graphic",E_Preferences,4
4,9887,1,2,1419247292,1419247292,True,48.47,0.4847,"A_Hardware,D_Browser,C_Audio","E_Preferences,B_Graphic",3
5,12841,0,1,1443069947,1443069947,True,77.08,0.7708,"A_Hardware,D_Browser,C_Audio,B_Graphic",E_Preferences,4
6,15780,0,1,1029125137,1437069253,False,0.00,0.0000,,"A_Hardware,D_Browser,E_Preferences,C_Audio,B_G...",0
7,15780,0,2,1029125137,1437069253,False,0.00,0.0000,,"A_Hardware,D_Browser,E_Preferences,C_Audio,B_G...",0
8,15780,1,2,1437069253,1437069253,True,77.08,0.7708,"A_Hardware,D_Browser,C_Audio,B_Graphic",E_Preferences,4
9,32191,0,1,681550052,1420908087,False,51.87,0.5187,"D_Browser,E_Preferences,C_Audio","A_Hardware,B_Graphic",3


## Test với Multiple Thresholds (80, 81, 82)


In [5]:
results = []

for threshold in TEST_THRESHOLDS:
    print(f"\n{'='*60}")
    print(f"THRESHOLD: {threshold} điểm")
    print(f"{'='*60}")
    
    # Predictions
    validation_df[f'predicted_same_device_{threshold}'] = validation_df['similarity_score'] >= threshold
    
    # Confusion Matrix
    predicted = validation_df[f'predicted_same_device_{threshold}']
    actual = validation_df['ground_truth_same_device']
    
    TP = ((predicted == True) & (actual == True)).sum()
    TN = ((predicted == False) & (actual == False)).sum()
    FP = ((predicted == True) & (actual == False)).sum()
    FN = ((predicted == False) & (actual == True)).sum()
    
    # Metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    
    # Rates
    false_positive_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
    false_negative_rate = FN / (FN + TP) if (FN + TP) > 0 else 0
    
    result = {
        'threshold': threshold,
        'TP': TP,
        'TN': TN,
        'FP': FP,
        'FN': FN,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'accuracy': accuracy,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate
    }
    
    results.append(result)
    
    print(f"\n📊 Confusion Matrix:")
    print(f"   TP (True Positive):  {TP:>8,}")
    print(f"   TN (True Negative):  {TN:>8,}")
    print(f"   FP (False Positive): {FP:>8,}")
    print(f"   FN (False Negative): {FN:>8,}")
    
    print(f"\n📈 Performance Metrics:")
    print(f"   Precision:  {precision*100:>6.2f}%")
    print(f"   Recall:     {recall*100:>6.2f}%")
    print(f"   F1-Score:   {f1_score:>6.4f}")
    print(f"   Accuracy:   {accuracy*100:>6.2f}%")
    
    print(f"\n📉 Error Rates:")
    print(f"   False Positive Rate: {false_positive_rate*100:>6.2f}%")
    print(f"   False Negative Rate:  {false_negative_rate*100:>6.2f}%")

results_df = pd.DataFrame(results)



THRESHOLD: 40 điểm

📊 Confusion Matrix:
   TP (True Positive):     1,038
   TN (True Negative):       738
   FP (False Positive):      207
   FN (False Negative):       16

📈 Performance Metrics:
   Precision:   83.37%
   Recall:      98.48%
   F1-Score:   0.9030
   Accuracy:    88.84%

📉 Error Rates:
   False Positive Rate:  21.90%
   False Negative Rate:    1.52%

THRESHOLD: 41 điểm

📊 Confusion Matrix:
   TP (True Positive):     1,038
   TN (True Negative):       738
   FP (False Positive):      207
   FN (False Negative):       16

📈 Performance Metrics:
   Precision:   83.37%
   Recall:      98.48%
   F1-Score:   0.9030
   Accuracy:    88.84%

📉 Error Rates:
   False Positive Rate:  21.90%
   False Negative Rate:    1.52%

THRESHOLD: 42 điểm

📊 Confusion Matrix:
   TP (True Positive):     1,037
   TN (True Negative):       781
   FP (False Positive):      164
   FN (False Negative):       17

📈 Performance Metrics:
   Precision:   86.34%
   Recall:      98.39%
   F1-Score:   0.91

## So Sánh Kết Quả Của 3 Thresholds


In [6]:
print("="*90)
print("RESULTS COMPARISON")
print("="*90)

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Accuracy':<12} {'FP Rate':<12} {'FN Rate':<12}")
print("-" * 90)

for _, row in results_df.iterrows():
    print(f"{row['threshold']:<12} {row['precision']*100:>10.2f}% {row['recall']*100:>10.2f}% "
          f"{row['f1_score']:>10.4f} {row['accuracy']*100:>10.2f}% "
          f"{row['false_positive_rate']*100:>10.2f}% {row['false_negative_rate']*100:>10.2f}%")

# Find best threshold
best_threshold_f1 = results_df.loc[results_df['f1_score'].idxmax()]

print(f"\n{'='*90}")
print(f"🎯 BEST THRESHOLD (F1-Score): {int(best_threshold_f1['threshold'])}")
print(f"{'='*90}")
print(f"   - F1-Score: {best_threshold_f1['f1_score']:.4f}")
print(f"   - Precision: {best_threshold_f1['precision']*100:.2f}%")
print(f"   - Recall: {best_threshold_f1['recall']*100:.2f}%")
print(f"   - Accuracy: {best_threshold_f1['accuracy']*100:.2f}%")


RESULTS COMPARISON

Threshold    Precision    Recall       F1-Score     Accuracy     FP Rate      FN Rate     
------------------------------------------------------------------------------------------
40.0              83.37%      98.48%     0.9030      88.84%      21.90%       1.52%
41.0              83.37%      98.48%     0.9030      88.84%      21.90%       1.52%
42.0              86.34%      98.39%     0.9197      90.95%      17.35%       1.61%
43.0              86.41%      98.29%     0.9197      90.95%      17.25%       1.71%
44.0              86.41%      98.29%     0.9197      90.95%      17.25%       1.71%
45.0              86.41%      98.29%     0.9197      90.95%      17.25%       1.71%
46.0              86.41%      98.29%     0.9197      90.95%      17.25%       1.71%
47.0              86.55%      98.29%     0.9205      91.05%      17.04%       1.71%
48.0              86.55%      98.29%     0.9205      91.05%      17.04%       1.71%
49.0              91.28%      94.40%     0

## Export Kết Quả


In [9]:
# Export results summary
results_df.to_csv('threshold_testing_results1.csv', index=False)
print("✓ Results summary exported to threshold_testing_results1.csv")

# Export detailed validation pairs
validation_export = validation_df[[
    'AccountID', 'fp1_DeviceCodeID', 'fp2_DeviceCodeID',
    'ground_truth_same_device', 'similarity_score', 'match_rate',
    'matched_groups', 'unmatched_groups', 'num_matched_groups'
] + [f'predicted_same_device_{t}' for t in TEST_THRESHOLDS]]

validation_export.to_csv('threshold_testing_validation_pairs1.csv', index=False)
print("✓ Validation pairs exported to threshold_testing_validation_pairs1.csv")

print(f"\n✅ HOÀN TẤT TESTING")
print(f"📁 Files exported:")
print(f"   1. threshold_testing_results1.csv")
print(f"   2. threshold_testing_validation_pairs1.csv")
print(f"   3. threshold_testing_results1.png")


✓ Results summary exported to threshold_testing_results1.csv
✓ Validation pairs exported to threshold_testing_validation_pairs1.csv

✅ HOÀN TẤT TESTING
📁 Files exported:
   1. threshold_testing_results1.csv
   2. threshold_testing_validation_pairs1.csv
   3. threshold_testing_results1.png


In [8]:
# results_df.to_csv('threshold_testing_validation_pairs.csv', index=False)